#  Intelligent LLM Router — Full Project Walkthrough
> **How to use this notebook:** Read every markdown cell carefully before running the code. Each section explains the *why* behind every decision, not just the *what*. This is your interview prep guide.

---

##  Project Architecture (Big Picture)

```
User Prompt
    │
    ▼
┌─────────────────────────────────────────┐
│           FastAPI Gateway               │
│  POST /api/chat/    GET /api/chat/stats │
└──────────────┬──────────────────────────┘
               │
        ┌──────▼──────┐
        │ LLM Router  │
        │  (llm_router.py)
        └──┬───┬───┬──┘
           │   │   │
    ┌──────┘   │   └──────┐
    ▼          ▼          ▼
[Cache?]  [Classify]  [Analyze        ]
[Hit]     [Task Type] [Complexity 0-1 ]
    │          │          │
    │          └────┬─────┘
    │               ▼
    │       ┌──────────────┐
    │       │ Select Model │
    │       │  8B / 70B-M  │
    │       │  / 70B-Heavy │
    │       └──────┬───────┘
    │              │
    │       ┌──────▼──────────┐
    │       │ groq_client.py  │
    │       │ (API call+time) │
    │       └──────┬──────────┘
    │              │
    │       ┌──────▼──────────┐
    │       │ Quality Gate    │
    │       │ (Thin? Retry↑)  │
    │       └──────┬──────────┘
    │              │
    └──────────────┘
               │
        ┌──────▼──────┐
        │  SQLite DB  │   ← logs every request
        └─────────────┘
               │
          JSON Response
```

---
###  Tech Stack Summary
| Component | Technology | Why |
|---|---|---|
| API Framework | FastAPI | Async, automatic Swagger docs, Pydantic validation |
| LLM Provider | Groq API | ~800 tokens/sec — fastest inference available |
| Models | Llama 3.1 8B / 3.1 70B / 3.3 70B | Open source, free tier, different capability tiers |
| Database | SQLite + SQLAlchemy | Zero-setup, perfect for local analytics logging |
| Validation | Pydantic v2 | Strict input/output typing, free OpenAPI docs |
| Server | Uvicorn | ASGI server, required by FastAPI |

---
# PART 1 — Environment Setup

## What is Groq?
Groq is **NOT** the same as xAI's Grok chatbot. Groq is a hardware+software company that built a custom chip called the **LPU (Language Processing Unit)** specifically for LLM inference. Their cloud API serves open-source models (Llama, Mistral, etc.) at ~800 tokens/second — roughly 10× faster than OpenAI.

**Why this matters for the project:** Speed is the whole value proposition of our router. By using Groq, even our "expensive" 70B model responds in under 3 seconds.

## Why `.env` files?
API keys are secrets. You never hardcode them in source code because:
- Code gets committed to GitHub → key gets exposed → key gets stolen
- `.env` file is added to `.gitignore` so it never leaves your machine
- `python-dotenv` reads it at runtime with `load_dotenv()`

**Interview answer:** _"I store credentials in environment variables using `.env` files and `python-dotenv`, which is the 12-Factor App best practice for configuration management."_

In [ ]:
# Install dependencies (run once)
# %pip install groq fastapi uvicorn python-dotenv sqlalchemy pydantic httpx==0.27.2

import os
from dotenv import load_dotenv

# load_dotenv() reads the .env file and injects its key=value pairs
# into the process environment variables (os.environ)
load_dotenv('../.env')  # Points to GenAI/.env

GROQ_API_KEY = os.getenv('GROQ_API_KEY')

# Verify it loaded (we print only first 10 chars for safety)
if GROQ_API_KEY:
    print(f'Key loaded: {GROQ_API_KEY[:12]}...')
else:
    print('ERROR: Key not found. Check your .env file path.')

---
# PART 2 — Groq Client (`groq_client.py`)

## What does this module do?
It acts as a **thin wrapper** around the Groq SDK. Its single responsibility is:
1. Accept a `prompt`, `model`, and optional `system_message`
2. Call the Groq API
3. Return the response text **plus** measured metrics (latency, token counts)

## Key Concepts

### System Message vs User Message
Chat LLMs take a **list of messages**, each with a `role`:
- `system` — Sets the AI's *persona and behavior*. Injected before the conversation.
- `user` — The actual question from the user.
- `assistant` — Previous AI responses (used for multi-turn chat)

```
messages = [
  {"role": "system",    "content": "You are an expert Python engineer..."},
  {"role": "user",      "content": "Write a linked list in Python"}
]
```

### Token Counting
LLMs are billed and limited by **tokens** (not words/characters). A token ≈ 4 characters on average.
- `prompt_tokens` — tokens we sent IN (our question + system message)
- `completion_tokens` — tokens the model generated (the answer)
- `total_tokens` — sum of both. This is what you would pay for with a paid API.

**Interview answer:** _"Tokens are the unit of measurement for LLM input/output. We track them per request to understand cost. For optimization, reducing prompt tokens means lower cost — that's one reason our router chooses a smaller model for simple tasks."_

In [ ]:
import time
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

def generate_completion(prompt: str, model: str, system_message: str = None):
    """
    Calls the Groq API and returns response text + metrics.
    
    Parameters:
        prompt         : The user's question
        model          : Which Groq model to use (e.g. 'llama-3.1-8b-instant')
        system_message : Optional persona/behavior instructions for the model
    """
    start_time = time.time()  # Start the stopwatch

    # Build the messages list — system message always comes first
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": prompt})

    # Make the API call
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=model,
    )

    latency_sec = time.time() - start_time  # Stop the stopwatch

    return {
        "response"          : chat_completion.choices[0].message.content,
        "model"             : model,
        "latency_sec"       : latency_sec,
        "prompt_tokens"     : chat_completion.usage.prompt_tokens,
        "completion_tokens" : chat_completion.usage.completion_tokens,
        "total_tokens"      : chat_completion.usage.total_tokens,
    }

# --- LIVE DEMO ---
result = generate_completion(
    prompt="What is 2+2?",
    model="llama-3.1-8b-instant",
    system_message="You are a concise math tutor."
)

print(f"Response    : {result['response']}")
print(f"Model       : {result['model']}")
print(f"Latency     : {result['latency_sec']:.3f}s")
print(f"Tokens Used : {result['total_tokens']} ({result['prompt_tokens']} in + {result['completion_tokens']} out)")

---
# PART 3 — Task Classifier (`classify_task`)

## What is it?
A **rule-based NLP classifier** that reads a prompt and assigns it one of four categories: `coding`, `math`, `creative`, or `factual`.

## How it works — Keyword Set Intersection
We maintain **sets** of keywords that belong to each domain. We then intersect the prompt's words with each set and score them.

```
prompt words = {"write", "python", "function", "binary", "search"}
coding_kw    = {"code", "function", "python", "algorithm", "binary", ...}

intersection = {"function", "python", "binary"} → score = 3  ← WINNER
```

## Why not use an LLM to classify?
Great interview question! We *could* call a small model to classify, but:
- That costs extra API time/money on EVERY request
- Keyword matching takes **microseconds** vs ~200ms for an LLM call
- For routing decisions, heuristics are good enough — we don't need 99% accuracy here

This trade-off (speed/cost vs accuracy) is called **latency vs quality** and is a key design consideration.

## Why does task type matter?
Different tasks need different *mindsets*. A coding prompt benefits from a system message like *"Write clean, production-quality code with examples"*, while a math prompt benefits from *"Show all working step-by-step"*. This is called **prompt engineering**.

In [ ]:
import re

TASK_SYSTEM_PROMPTS = {
    "coding"  : "You are an expert software engineer. Write clean, production-quality code with usage examples.",
    "math"    : "You are an expert mathematician. Solve step-by-step, showing all reasoning.",
    "creative": "You are a creative writing expert. Be vivid, original, and engaging.",
    "factual" : "You are a knowledgeable assistant. Answer concisely and accurately.",
}

def classify_task(prompt: str) -> str:
    lower = prompt.lower()
    
    coding_kw   = {"code","script","function","class","debug","refactor","implement",
                   "python","javascript","sql","api","algorithm","binary","array",
                   "tree","graph","react","fastapi","compile","runtime"}
    math_kw     = {"calculate","solve","prove","theorem","equation","integral",
                   "matrix","probability","statistics","limit","formula","algebra",
                   "calculus","optimize","derivative"}
    creative_kw = {"write","poem","story","essay","creative","fiction","narrative",
                   "character","plot","scene","describe","imagine","draft"}

    words = set(re.findall(r'\b\w+\b', lower))

    scores = {
        "coding"  : len(words & coding_kw),
        "math"    : len(words & math_kw),
        "creative": len(words & creative_kw),
    }
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else "factual"


# --- TEST IT ---
test_prompts = [
    "Write a Python function for binary search",
    "Calculate the derivative of sin(x)",
    "Write a short poem about the ocean",
    "What is the capital of Japan?",
]

for p in test_prompts:
    task = classify_task(p)
    print(f"  [{task.upper():8s}] {p}")

---
# PART 4 — Complexity Analyzer (`analyze_complexity`)

## What is it?
A **multi-factor scoring function** that returns a float between `0.0` and `1.0` representing how hard a prompt is. This score determines which model tier we route to.

## The 5 Factors (and why each matters)

| Factor | Logic | Rationale |
|---|---|---|
| **Length** | >150 words → +0.35 | Long prompts usually contain multi-step instructions |
| **Action keywords** | "analyze", "design", "prove" → +0.20 | These words signal complex reasoning tasks |
| **Technical terms** | "neural", "distributed", "backpropagation" → +0.10 each | Domain jargon correlates with complexity |
| **Code blocks** | ` ``` ` in prompt → +0.20 | A code block means they want code back — use a code-savvy model |
| **Multi-part** | Numbered lists or bullet points → +0.10 | Multiple sub-questions = more complex |

## Why not use a single threshold?
Real prompts don't fit into "simple" or "complex" buckets. A 3-factor weighted score captures nuance. For example:
- "What is React?" → short, one tech word → score 0.10 → fast model
- "Explain React reconciliation algorithm vs. Vue's virtual DOM diffing" → medium length + action words + tech terms → score ~0.45 → medium model
- "Write a distributed job queue in Python using Redis and asyncio, including error handling and retry logic" → long + action + many tech terms + code expected → score 0.85+ → heavy model

**Interview answer:** _"Our complexity scorer is a heuristic multi-factor function. I chose heuristics over ML classification because the ground truth for 'complexity' is subjective, we had no labeled training data, and rule-based scoring is fast, explainable, and easy to iterate on."_

In [ ]:
def analyze_complexity(prompt: str) -> float:
    score = 0.0
    lower = prompt.lower()

    # Factor 1: Length
    words = len(prompt.split())
    if words > 150:    score += 0.35
    elif words > 75:   score += 0.20
    elif words > 30:   score += 0.05

    # Factor 2: Action keywords
    action_kw = {"analyze","compare","design","architect","implement",
                 "refactor","debug","optimize","calculate","prove"}
    if any(kw in lower for kw in action_kw):
        score += 0.20

    # Factor 3: Technical term density (each term adds 0.10, capped at 0.30)
    technical_kw = {
        "algorithm","neural","transformer","distributed","concurrency",
        "microservice","machine learning","gradient","backpropagation",
        "dynamic programming","recursion","async"
    }
    tech_matches = sum(1 for kw in technical_kw if kw in lower)
    score += min(0.30, tech_matches * 0.10)

    # Factor 4: Code blocks in prompt
    if "```" in prompt: score += 0.20

    # Factor 5: Multi-part questions
    if re.search(r'(\d+[\.)\s]|\* |- )', prompt): score += 0.10

    return min(1.0, round(score, 3))


# --- SCORE DISTRIBUTION DEMO ---
test_cases = [
    ("What is the capital of France?", "Simple factual"),
    ("Write a Python binary search function", "Simple coding"),
    ("Compare React and Vue's virtual DOM diffing algorithms", "Medium technical"),
    ("Design a distributed job queue in Python using asyncio and Redis with retry logic and dead-letter queues", "Heavy architecture"),
    ("Implement a transformer neural network from scratch using backpropagation and gradient descent", "Very heavy ML"),
]

print(f"{'Score':>6}  {'Category':<18}  Prompt")
print("-" * 75)
for prompt, category in test_cases:
    score = analyze_complexity(prompt)
    tier  = "8B FAST" if score < 0.25 else ("70B MED" if score < 0.55 else "70B HEAVY")
    print(f"  {score:.3f}  [{tier:9s}]  {prompt[:55]}")

---
# PART 5 — Three-Tier Model Routing

## The Model Tiers
| Score | Model | Use Case | Approx. Speed |
|---|---|---|---|
| `< 0.25` | `llama-3.1-8b-instant` | Factual Q&A, quick lookups | ~0.2–0.5s |
| `0.25–0.54` | `llama-3.1-70b-versatile` | Technical explanations, analysis | ~0.5–2s |
| `≥ 0.55` | `llama-3.3-70b-versatile` | Complex code, architecture, research | ~1–4s |

## Why 3 tiers and not just 2?
With 2 tiers (fast/slow), there's a binary jump in cost. Many real-world prompts are "medium complexity" — technical questions that don't need the heaviest model but are too involved for the 8B. The middle tier (3.1-70B) provides a **good balance of capability and speed** at lower inference cost.

## Why are these specific thresholds (0.25 and 0.55)?
Chosen empirically by testing a range of prompts. The key insight is that anything below 0.25 is so simple (~1-2 factor matches) that the 8B handles it perfectly.

In [ ]:
FAST_MODEL   = "llama-3.1-8b-instant"
MEDIUM_MODEL = "llama-3.1-70b-versatile"
HEAVY_MODEL  = "llama-3.3-70b-versatile"

def select_model(score: float) -> str:
    """Pure function: score → model name. No side effects."""
    if score >= 0.55:
        return HEAVY_MODEL
    elif score >= 0.25:
        return MEDIUM_MODEL
    else:
        return FAST_MODEL

# Visualize the decision boundary
print("Score → Tier Mapping")
print("─" * 40)
for score in [0.0, 0.10, 0.24, 0.25, 0.40, 0.54, 0.55, 0.80, 1.0]:
    model = select_model(score)
    tier  = model.split('-')[2].upper() + ' ' + model.split('-')[3]
    print(f"  {score:.2f}  →  {model}")

---
# PART 6 — Quality Gatekeeper (Auto-Retry)

## Problem it solves
Sometimes a small 8B model gives a **too-short answer** for a question it should be capable of answering. For example, asking "What is the capital of France?" might return just "Paris." — 1 word. The user wanted a full sentence.

## Solution: Escalation on Thin Responses
After getting a response from the **fast model only**, we check if the response is less than 12 words. If yes, we automatically re-run the request on the **heavy model** and return that instead.

## Why 12 words as the threshold?
Empirical choice. Most coherent sentences are at least 10-15 words. A one-word or one-sentence response from a factual question usually means the model took a shortcut.

## Important design decision: Only retry from FAST model
We **don't** retry from the medium or heavy model because:
- If the heavy model gives a short answer, it was intentional (the answer IS short)
- Retrying from the heavy model would double the cost of the most expensive tier

**Interview answer:** _"The quality gatekeeper is a simple post-processing step that catches under-answered responses from the fast model. It's a pragmatic trade-off: we accept slightly higher latency on a subset of fast-tier queries in exchange for better answer quality without increasing average cost."_

In [ ]:
def _is_response_thin(text: str, min_words: int = 12) -> bool:
    """Returns True if the response is suspiciously short."""
    return len(text.split()) < min_words

# Simulate the logic
responses = [
    ("Paris.",                                    True,  "Too short — would escalate"),
    ("The capital of France is Paris.",            False, "OK — 7 words but borderline"),
    ("The capital of France is Paris, which has been the capital since the 10th century.", False, "OK — full answer"),
    ("4",                                          True,  "Too short — would escalate"),
]

print(f"{'Thin?':^6}  {'Expected':^5}  Response")
print("-" * 70)
for response, expected, note in responses:
    thin   = _is_response_thin(response)
    match  = "OK" if thin == expected else "MISMATCH"
    print(f"  {'YES' if thin else 'NO ':3s}    {match:8s}  '{response[:50]}' ({note})")

---
# PART 7 — In-Memory Caching

## Problem it solves
If the same prompt is sent twice, there's no reason to call the API again — we already know the answer. API calls cost money and take time.

## Implementation: SHA-256 Hash as Cache Key
We use a Python dictionary `{}` as the cache. The key is the **SHA-256 hash** of the normalized prompt (lowercased, stripped).

**Why SHA-256 and not the raw string?**
- Hashes are fixed-length (64 chars) regardless of prompt size → efficient dict key
- Normalizing (lowercase + strip) before hashing means "What is 2+2?" and "what is 2+2?" hash identically

## Limitations of this approach
This is an **in-memory** cache — it lives in RAM and disappears when the server restarts. A production system would use **Redis** for a persistent, shared cache across multiple server instances.

**Interview answer:** _"We implemented a simple in-process dict cache using SHA-256 hashed keys for O(1) lookup. For production, this would be replaced with Redis to support distributed caching across multiple server replicas."_

In [ ]:
import hashlib

_prompt_cache = {}

def _cache_key(prompt: str) -> str:
    """Normalize prompt → SHA-256 hash → use as cache key."""
    normalized = prompt.strip().lower()
    return hashlib.sha256(normalized.encode()).hexdigest()

# --- DEMO ---
p1 = "What is the capital of France?"
p2 = "  what is the capital of france?  "  # Same, but with spaces and different case
p3 = "What is the capital of Germany?"

print("Prompt 1 key:", _cache_key(p1)[:16], "...")
print("Prompt 2 key:", _cache_key(p2)[:16], "...")
print("Prompt 3 key:", _cache_key(p3)[:16], "...")
print()
print(f"p1 == p2: {_cache_key(p1) == _cache_key(p2)}  ← same key despite spacing/case differences")
print(f"p1 == p3: {_cache_key(p1) == _cache_key(p3)}  ← different prompts, different keys")

---
# PART 8 — Database Logging (SQLite + SQLAlchemy)

## Why log every request?
Logging is what separates a **toy project** from a **production system**. With the database we can:
- See which models get chosen most often
- Track average latency over time
- Count how many times the quality gatekeeper fires
- Analyze cost (token usage) by task type

## What is SQLAlchemy?
SQLAlchemy is Python's most popular **ORM (Object-Relational Mapper)**. An ORM lets you define database tables as Python classes and interact with the database using Python objects instead of raw SQL strings.

```
# Without ORM (raw SQL — error-prone, hard to maintain):
cursor.execute("INSERT INTO request_logs (prompt, model ...) VALUES (?, ?, ...)", (...))

# With SQLAlchemy ORM (Pythonic, safe, readable):
log = RequestLog(prompt=prompt, model=model, ...)
db.add(log)
db.commit()
```

## Why SQLite for a project like this?
- **Zero setup** — it's a single `.db` file on disk. No separate database server needed.
- **Perfect for single-machine projects** — our router runs on one machine.
- **Production upgrade path**: Just change `SQLALCHEMY_DATABASE_URL = "postgresql://..."` and the rest of the code stays identical. That's the power of using an ORM abstraction.

**Interview answer:** _"I used SQLite for local development because it requires no configuration and SQLAlchemy's ORM abstraction means migrating to PostgreSQL in production is a single connection string change."_

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, Boolean, DateTime, func, case, text
from sqlalchemy.orm import declarative_base, sessionmaker
import datetime

Base = declarative_base()

# --- The Database Table Definition ---
# Each class attribute = one column in the SQL table
class RequestLog(Base):
    __tablename__ = "request_logs"

    id                = Column(Integer, primary_key=True, index=True)
    timestamp         = Column(DateTime, default=datetime.datetime.utcnow)
    prompt            = Column(String)
    task_type         = Column(String, index=True)      # 'coding'|'math'|'creative'|'factual'
    complexity_score  = Column(Float)
    chosen_model      = Column(String, index=True)
    latency_sec       = Column(Float)
    prompt_tokens     = Column(Integer)
    completion_tokens = Column(Integer)
    total_tokens      = Column(Integer)
    retried           = Column(Boolean, default=False)  # Did quality gate fire?
    from_cache        = Column(Boolean, default=False)  # Was this answered from cache?

# --- Create an in-memory SQLite DB for demo ---
engine     = create_engine("sqlite:///:memory:")
Base.metadata.create_all(bind=engine)
Session    = sessionmaker(bind=engine)
db         = Session()

# --- Insert some fake log entries ---
fake_logs = [
    RequestLog(prompt="What is 2+2?", task_type="factual", complexity_score=0.0,
               chosen_model="llama-3.1-8b-instant", latency_sec=0.21,
               prompt_tokens=40, completion_tokens=5, total_tokens=45, retried=True),
    RequestLog(prompt="Write a BST in Python", task_type="coding", complexity_score=0.2,
               chosen_model="llama-3.1-8b-instant", latency_sec=1.5,
               prompt_tokens=65, completion_tokens=1200, total_tokens=1265, retried=False),
    RequestLog(prompt="Explain transformer architecture", task_type="coding", complexity_score=0.6,
               chosen_model="llama-3.3-70b-versatile", latency_sec=2.1,
               prompt_tokens=80, completion_tokens=700, total_tokens=780, retried=False),
]
for log in fake_logs:
    db.add(log)
db.commit()

# --- Query: What the /stats endpoint returns ---
rows = (
    db.query(
        RequestLog.chosen_model,
        func.count(RequestLog.id).label("total_requests"),
        func.round(func.avg(RequestLog.latency_sec), 3).label("avg_latency"),
        func.sum(RequestLog.total_tokens).label("total_tokens"),
        func.sum(case((RequestLog.retried == True, 1), else_=0)).label("retries"),
    )
    .group_by(RequestLog.chosen_model)
    .all()
)

print(f"{'Model':<35} {'Reqs':>4} {'AvgLat':>8} {'Tokens':>8} {'Retries':>8}")
print("-" * 70)
for r in rows:
    print(f"{r.chosen_model:<35} {r.total_requests:>4} {r.avg_latency:>8.3f}s {r.total_tokens:>8} {r.retries:>8}")

---
# PART 9 — FastAPI: Pydantic, Routing, Dependency Injection

## What is FastAPI?
FastAPI is a modern Python web framework built on top of Starlette (ASGI) and Pydantic. Its key features:
- **Automatic validation** via Pydantic models — invalid requests return a proper 422 error automatically
- **Automatic OpenAPI (Swagger) docs** at `/docs` — zero extra work needed
- **Dependency injection** — pass shared resources (like DB sessions) into endpoints cleanly
- **Async-first** — handles many requests concurrently without blocking

## What is Pydantic?
A Python library for **data validation using type hints**. You define a model class and Pydantic will:
1. Parse incoming JSON into that class
2. Validate types (strings are strings, floats are floats)
3. Raise a clean error if data doesn't match the schema
4. Automatically generate JSON Schema (used for Swagger docs)

## What is Dependency Injection (`Depends`)?
Instead of creating a new database session inside every endpoint function, FastAPI's `Depends()` creates and manages the session for you. It:
- Creates the DB session before the endpoint runs
- Closes/cleans it up after the endpoint returns (even on errors)
- Makes testing easy — you can swap in a test DB session

**Interview answer:** _"FastAPI uses dependency injection via `Depends()` to manage resources like database sessions. This pattern separates resource lifecycle management from business logic, making the code more testable and preventing connection leaks."_

In [ ]:
from pydantic import BaseModel

# --- Input schema (what the user sends) ---
class ChatRequest(BaseModel):
    prompt: str

# --- Output schema (what we send back) ---
class ChatResponse(BaseModel):
    response:          str
    model:             str
    latency_sec:       float
    prompt_tokens:     int
    completion_tokens: int
    total_tokens:      int
    complexity_score:  float
    task_type:         str
    retried:           bool
    from_cache:        bool

# --- Demo: Pydantic validation in action ---
valid_input   = ChatRequest(prompt="Hello, world!")
print("Valid:  ", valid_input)

try:
    # Missing required 'prompt' field
    bad_input = ChatRequest()   # type: ignore
except Exception as e:
    print("Invalid:", type(e).__name__, "— FastAPI would auto-return a 422 Unprocessable Entity")

print()
print("JSON Schema (used to generate Swagger /docs):")
import json
print(json.dumps(ChatRequest.model_json_schema(), indent=2))

---
# PART 10 — Full Pipeline: Putting It All Together

## The complete `route_request` function flow

```
route_request(prompt)
  │
  ├─1─ Check cache (sha256 dict lookup)  ──── HIT ──► return cached result immediately
  │                                           (zero API cost)
  │         MISS
  │
  ├─2─ classify_task(prompt)            ──► 'coding' | 'math' | 'creative' | 'factual'
  │         ↓
  │    system_msg = TASK_SYSTEM_PROMPTS[task_type]
  │
  ├─3─ analyze_complexity(prompt)       ──► float 0.0 → 1.0
  │         ↓
  │    select_model(score)              ──► 8B | 70B-Med | 70B-Heavy
  │
  ├─4─ generate_completion(prompt, model, system_msg)
  │         ↓
  │    Groq API call → response text + tokens + latency
  │
  ├─5─ Quality Gate (FAST model only)
  │    if response is thin (<12 words):
  │         re-run with HEAVY_MODEL
  │         set retried = True
  │
  ├─6─ Store in cache
  │
  └─7─ Return full result dict
```

In [ ]:
# Complete self-contained pipeline (no imports from router_project)

def route_request_demo(prompt: str, verbose: bool = True) -> dict:
    """Full pipeline from prompt to result."""
    key = _cache_key(prompt)

    # Step 1: Cache check
    if key in _prompt_cache:
        if verbose: print(f"  [CACHE HIT] Returning cached result for: {prompt[:40]}")
        return {**_prompt_cache[key], "from_cache": True}

    if verbose: print(f"  [CACHE MISS] Processing fresh: '{prompt[:50]}'")

    # Step 2: Classify task → get system message
    task_type  = classify_task(prompt)
    system_msg = TASK_SYSTEM_PROMPTS[task_type]
    if verbose: print(f"  [CLASSIFY]  Task type: {task_type.upper()}")

    # Step 3: Score complexity → select model
    score = analyze_complexity(prompt)
    model = select_model(score)
    if verbose: print(f"  [COMPLEXITY] Score: {score} → Model: {model}")

    # Step 4: API call
    if verbose: print(f"  [API CALL]  Sending to Groq...")
    result = generate_completion(prompt, model, system_msg)

    # Step 5: Quality gate
    retried = False
    if model == FAST_MODEL and _is_response_thin(result["response"]):
        if verbose: print(f"  [RETRY]     Response too thin! Escalating to {HEAVY_MODEL}")
        result  = generate_completion(prompt, HEAVY_MODEL, system_msg)
        retried = True

    result.update({"complexity_score": score, "task_type": task_type,
                   "retried": retried, "from_cache": False})

    # Step 6: Cache
    _prompt_cache[key] = result

    if verbose:
        print(f"  [DONE]      '{result['response'][:60]}...'" if len(result['response']) > 60 else f"  [DONE]      '{result['response']}'")
        print(f"              Latency: {result['latency_sec']:.2f}s | Tokens: {result['total_tokens']}")

    return result


# --- LIVE TEST ---
print("=" * 60)
print("TEST 1: Simple factual question")
print("=" * 60)
r = route_request_demo("What is the capital of France?")

print()
print("=" * 60)
print("TEST 2: Same question again (should hit cache)")
print("=" * 60)
r = route_request_demo("What is the capital of France?")

print()
print("=" * 60)
print("TEST 3: Complex coding question")
print("=" * 60)
r = route_request_demo("Implement a distributed rate limiter in Python using Redis and asyncio with sliding window algorithm")

---
# PART 11 — Running the Server

## FastAPI App Structure Recap

```
router_project/
├── app/
│   ├── main.py                   ← FastAPI app creation, middleware, router inclusion
│   ├── api/endpoints/chat.py     ← POST /api/chat/ and GET /api/chat/stats endpoints
│   ├── schemas/chat.py           ← Pydantic models (ChatRequest, ChatResponse)
│   ├── services/
│   │   ├── groq_client.py        ← Groq SDK wrapper + latency measurement
│   │   └── llm_router.py         ← Core routing intelligence
│   └── db/
│       ├── database.py           ← SQLAlchemy engine, session, init_db()
│       └── models.py             ← RequestLog ORM model
└── requirements.txt
```

## What is Uvicorn?
Uvicorn is an **ASGI server** — it sits in front of your FastAPI app and handles all the low-level HTTP stuff (accepting TCP connections, parsing HTTP headers, etc.).

- **WSGI** (old, sync) — used by Flask, Django
- **ASGI** (new, async) — used by FastAPI, Starlette. Supports WebSockets and true async.

`--reload` flag watches your files for changes and automatically restarts the server — perfect for development.

## API Endpoints Summary
| Method | Path | What it does |
|---|---|---|
| `POST` | `/api/chat/` | Send a prompt, get a routed LLM response |
| `GET` | `/api/chat/stats` | Get per-model aggregate analytics from the DB |
| `GET` | `/docs` | Auto-generated Swagger UI (free with FastAPI!) |
| `GET` | `/` | Health check — returns `{"message": "running"}` |

In [ ]:
# ─── How to start the server ───────────────────────────────────────────────
# Run these commands in your terminal (NOT in notebook)

startup_commands = """
# Navigate to the router project
cd Desktop/GenAI/router_project

# Activate virtual environment
source venv/bin/activate

# Start the server with hot-reload
uvicorn app.main:app --reload --port 8000

# Open: http://localhost:8000/docs  — Swagger UI
# Open: http://localhost:8000/redoc — ReDoc UI (alternate docs)
"""

print(startup_commands)

# ─── How to test the API with curl ─────────────────────────────────────────
curl_examples = """
# Simple query (routes to 8B fast model)
curl -X POST http://localhost:8000/api/chat/ \\
  -H 'Content-Type: application/json' \\
  -d '{"prompt": "What is Python?"}'

# Complex coding query (routes to 70B heavy model)
curl -X POST http://localhost:8000/api/chat/ \\
  -H 'Content-Type: application/json' \\
  -d '{"prompt": "Design a distributed caching system with Redis and asyncio"}'

# Get analytics stats
curl http://localhost:8000/api/chat/stats
"""

print(curl_examples)

---
# PART 12 — Interview Cheat Sheet

## Core Questions You Must Be Able to Answer

---
### Q: "What does your router do at a high level?"
> *"It's an intelligent API gateway. Instead of always sending prompts to the most expensive LLM, it analyzes each prompt's complexity using a multi-factor scoring function and routes it to one of three model tiers — a fast 8B model for simple tasks, a medium 70B model for technical questions, and a heavy 70B model for complex architecture or ML tasks. This balances cost and quality automatically."*

---
### Q: "Why FastAPI over Flask?"
> *"FastAPI is async-first and uses Pydantic for automatic request validation, which means invalid inputs return proper 422 errors without writing any extra code. It also auto-generates OpenAPI (Swagger) documentation. Flask is sync and requires external libraries for these features."*

---
### Q: "How does your complexity analyzer work?"
> *"It's a heuristic multi-factor scoring function that checks five signals: prompt length, presence of action keywords like 'analyze' or 'design', technical term density (neural networks, distributed systems, etc.), presence of code blocks, and multi-part question structure. Each factor adds to a float score between 0 and 1, which maps to a model tier."*

---
### Q: "Why heuristics and not ML for classification?"
> *"We had no labeled training data for 'prompt complexity'. Heuristics are interpretable, fast (microseconds), and easy to tune. An ML classifier would require data collection, training, and ongoing maintenance — disproportionate effort for a routing signal that doesn't need to be 99% accurate."*

---
### Q: "What is your caching strategy?"
> *"In-memory dictionary with SHA-256 hashed and normalized prompt keys. The normalization (lowercase + strip) ensures semantically identical prompts with different formatting don't duplicate cache entries. The limitation is it's process-local — a restart clears it. For production, I'd use Redis."*

---
### Q: "What is dependency injection in FastAPI?"
> *"It's a pattern where shared resources like database sessions are provided to endpoints by FastAPI's `Depends()` decorator. The function declared in `Depends` runs before each request, yields the resource, and runs cleanup code after — even if the request fails. This prevents resource leaks."*

---
### Q: "What would you improve if this were a production system?"
> *"Four things: (1) Replace the in-memory cache with Redis for durability and distributed caching. (2) Replace SQLite with PostgreSQL for concurrent writes. (3) Add async LLM calls to handle concurrent requests without blocking. (4) Add circuit breaker logic — if Groq is down, fall back gracefully with a proper 503 response."*

---
### Q: "What is an ORM and why use one?"
> *"An ORM (Object-Relational Mapper) lets you interact with databases using Python objects instead of raw SQL strings. We used SQLAlchemy. The key benefit is the database abstraction — I can switch from SQLite to PostgreSQL by changing a single connection string."*

---
### Q: "What is the quality gatekeeper?"
> *"After getting a response from the fast 8B model, we check if it's less than 12 words. If yes, we automatically retry with the heavy 70B model. This handles cases where the small model gives a valid but too-brief answer. We only apply this to the fast model — if the heavy model gives a short answer, it was intentional."*

---
##  Concepts to Review Before Interview
- REST API methods: GET, POST, PUT, DELETE — and when to use each
- HTTP status codes: 200 OK, 201 Created, 400 Bad Request, 422 Unprocessable, 500 Internal Error
- What CORS is and why we needed `allow_origins=["*"]` for the frontend
- What ASGI vs WSGI means
- Time complexity of a dict lookup: O(1)
- What SHA-256 is: a deterministic hash function (same input → always same output)
- What an environment variable is and why secrets should live there
- What a token is in LLM context